In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
data_dir = '../../../data/processed/Misar-E13-E15-E18'

ad_mult_rna = sc.read_h5ad(join(data_dir, 'E15/ad_rna.h5ad'))
ad_mult_atac = sc.read_h5ad(join(data_dir, 'E15/ad_atac.h5ad'))

ad_test_rna = sc.read_h5ad(join(data_dir, 'E13/ad_rna.h5ad'))
ad_test_atac = sc.read_h5ad(join(data_dir, 'E18/ad_atac.h5ad'))

input_dict = { 
    'rna':   [ad_mult_rna, ad_test_rna, None],
    'atac':  [ad_mult_atac,None,        ad_test_atac],
}

input_key = 'dimred_bc'
batch_key = 'Sample'

In [3]:
Epigenome_preprocess(input_dict['atac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
RNA_preprocess(input_dict['rna'], batch_corr=True, n_hvg=10000, batch_key=batch_key, key=input_key)

Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Reach convergence after 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).


In [4]:
def stack(xl, key):
    xs, ns = [], []
    for adx in xl:
        if adx is not None:
            xs.append(adx.obsm[key])
            ns.append(adx.obs_names)
    df = pd.DataFrame(np.vstack(xs), index=np.hstack(ns))
    return df

for m1, m2 in zip(['rna', 'atac'], ['RNA', 'ATAC']):
    fig_dir = f'../../../results/embeddings/Leiden-{m2}/Misar-E13-E15-E18'
    os.makedirs(fig_dir, exist_ok=True)
    df = stack(input_dict[m1], input_key)
    df.to_csv(join(fig_dir, 'df_emb.csv'))